# QLR Feasibility over the Weyl Chamber

**Goal**: For a grid of Weyl chamber points, check which triples $(C, G, T)$ are QLR-feasible, and verify that feasibility is symmetric under permutation of the three roles.

The QLR inequalities for a single segment are:
$$L_C \cdot m_C + L_G \cdot m_G + L_T \cdot m_T \leq b$$

where $m_C, m_G, m_T$ are monodromy coordinates and $L_C, L_G, L_T$ are the 72×3 QLR blocks. With all three points fixed, this is a pure linear feasibility check (no LP needed).

In [16]:
import numpy as np
from itertools import permutations, product
from gulps._accelerate import monodromy_from_weyl as _mono
from gulps.linear_program.qlr import qlr_inequalities

_ci_block, _gi_block, _ciplus1_block, _bi = qlr_inequalities


def qlr_feasible(m_c, m_g, m_t, tol=1e-12):
    """Check 72 QLR inequalities for a single (prefix, gate, target) triple.

    All inputs are monodromy 3-vectors.
    Returns True if all 72 inequalities are satisfied (within tolerance).
    """
    lhs = _ci_block @ m_c + _gi_block @ m_g + _ciplus1_block @ m_t
    return bool(np.all(lhs <= _bi + tol))


# Vectorized version for bulk checking
def qlr_feasible_bulk(M_c, M_g, M_t, tol=1e-12):
    """Check QLR feasibility for arrays of triples.

    M_c, M_g, M_t: (N, 3) arrays of monodromy coordinates.
    Returns: (N,) boolean array.
    """
    # (72, 3) @ (3, N) -> (72, N)
    lhs = _ci_block @ M_c.T + _gi_block @ M_g.T + _ciplus1_block @ M_t.T
    return np.all(lhs <= _bi[:, None] + tol, axis=0)


print(f"QLR system: {_ci_block.shape[0]} inequalities, {_ci_block.shape[1]}D per role")
print(
    f"Sanity: (I, I, I) feasible = {qlr_feasible(np.zeros(3), np.zeros(3), np.zeros(3))}"
)

QLR system: 72 inequalities, 3D per role
Sanity: (I, I, I) feasible = True


## Weyl Chamber Grid

Use `weyl_linspace(N)` to generate well-spread sample points inside the canonical Weyl chamber tetrahedron (vertices → centroid → barycentric refinement). Then convert to monodromy coordinates.

In [17]:
from gulps.core.coverage import weyl_linspace
from qiskit.quantum_info.random import random_unitary
from gulps import GateInvariants


N_WEYL = int(np.sqrt(128 * 256))  # number of Weyl chamber sample points


def random_sample(n):
    """Generate n random points in monodromy space."""
    for _ in range(n):
        U = random_unitary(4).data
        yield GateInvariants.from_unitary(U).monodromy


weyl_points = np.array(list(weyl_linspace(N_WEYL)))
mono_points = np.array([_mono(c1, c2, c3) for c1, c2, c3 in weyl_points])
mono_points_random = np.array(list(random_sample(N_WEYL)))
# mono_points = mono_points_random  # switch to random sampling if desired

print(f"Weyl grid: {len(weyl_points)} points (requested {N_WEYL})")
print(f"Corners: I=(0,0,0), CX=(1,0,0), iSWAP=(0.5,0.5,0), SWAP=(0.5,0.5,0.5)")
print(f"First 5 points:\n{weyl_points[:5]}")
print(f"Monodromy shape: {mono_points.shape}")

Weyl grid: 181 points (requested 181)
Corners: I=(0,0,0), CX=(1,0,0), iSWAP=(0.5,0.5,0), SWAP=(0.5,0.5,0.5)
First 5 points:
[[0.    0.    0.   ]
 [1.    0.    0.   ]
 [0.5   0.5   0.   ]
 [0.5   0.5   0.5  ]
 [0.5   0.25  0.125]]
Monodromy shape: (181, 3)


## Feasibility Scan

Check all $N^3$ triples. For each triple $(C, G, T)$, the QLR check is just 72 dot products — no LP solver needed.

In [18]:
import time

N = len(mono_points)
print(f"Checking {N}^3 = {N**3:,} triples...")

# Build all triples via meshgrid indices
idx = np.arange(N)
i_c, i_g, i_t = np.meshgrid(idx, idx, idx, indexing="ij")
i_c, i_g, i_t = i_c.ravel(), i_g.ravel(), i_t.ravel()

M_c = mono_points[i_c]  # (N^3, 3)
M_g = mono_points[i_g]
M_t = mono_points[i_t]

t0 = time.perf_counter()
feasible = qlr_feasible_bulk(M_c, M_g, M_t)
dt = time.perf_counter() - t0

n_feasible = feasible.sum()
print(
    f"Feasible: {n_feasible:,} / {len(feasible):,} ({100 * n_feasible / len(feasible):.1f}%)"
)
print(f"Time: {dt:.3f}s")

# Reshape to 3D tensor for permutation analysis
feas_tensor = feasible.reshape(N, N, N)
print(f"Feasibility tensor shape: {feas_tensor.shape}")

Checking 181^3 = 5,929,741 triples...
Feasible: 1,629,001 / 5,929,741 (27.5%)
Time: 2.300s
Feasibility tensor shape: (181, 181, 181)


## Permutation Symmetry Test

The QLR encodes $U_C \cdot U_G = U_T$, equivalently $C \cdot G \cdot T^\dagger = I$.

### Dagger vs Rho-reflect

**Critical distinction**: the correct monodromy-level inverse is the **true dagger**, NOT the rho-reflect.

| Operation | Logspec | Monodromy | Weyl |
|-----------|---------|-----------|------|
| **Dagger** $(U^\dagger)$ | $(a,b,c,d) \to (-d,-c,-b,-a)$ | $(m_0,m_1,m_2) \to (m_0{+}m_1{+}m_2,\; -m_2,\; -m_1)$ | $(c_1,c_2,c_3) \to (1{-}c_1, c_2, c_3)$ |
| **Rho-reflect** | $(a,b,c,d) \to (c{+}\tfrac{1}{2}, d{+}\tfrac{1}{2}, a{-}\tfrac{1}{2}, b{-}\tfrac{1}{2})$ | affine | $(c_1,c_2,c_3) \to (1{-}c_1, c_2, -c_3)$ |

They differ by a $c_3$ sign flip and only agree when $c_3 = 0$.

### Horn symmetry

The quantum Horn problem (Agnihotri–Woodward) guarantees $S_3$ symmetry for the triple $(A, B, C^\dagger)$ where $A \cdot B = C$. The 72 QLR inequalities are the complete set of quantum Horn inequalities for SU(4). Permutation symmetry applies to the three Horn factors $\{C, G, T^\dagger\}$:

| Product | Factors | `L(pre, gate, target)` |
|---------|---------|----------------------|
| $T$ | $C \cdot G$ | $L(C, G, T)$ |
| $T$ | $G \cdot C$ | $L(G, C, T)$ |
| $G^\dagger$ | $C \cdot T^\dagger$ | $L(C, T^\dagger, G^\dagger)$ |
| $G^\dagger$ | $T^\dagger \cdot C$ | $L(T^\dagger, C, G^\dagger)$ |
| $C^\dagger$ | $G \cdot T^\dagger$ | $L(G, T^\dagger, C^\dagger)$ |
| $C^\dagger$ | $T^\dagger \cdot G$ | $L(T^\dagger, G, C^\dagger)$ |
| $G$ | $C^\dagger \cdot T$ | $L(C^\dagger, T, G)$ |
| $G$ | $T \cdot C^\dagger$ | $L(T, C^\dagger, G)$ |
| $C$ | $G^\dagger \cdot T$ | $L(G^\dagger, T, C)$ |
| $C$ | $T \cdot G^\dagger$ | $L(T, G^\dagger, C)$ |
| $T^\dagger$ | $G^\dagger \cdot C^\dagger$ | $L(G^\dagger, C^\dagger, T^\dagger)$ |
| $T^\dagger$ | $C^\dagger \cdot G^\dagger$ | $L(C^\dagger, G^\dagger, T^\dagger)$ |

In [19]:
from gulps.core.invariants import GateInvariants


def true_dagger_mono(m):
    """True monodromy inverse: (m0,m1,m2) → (m0+m1+m2, -m2, -m1).

    Derivation: logspec (a,b,c,d) with a≥b≥c≥d, a+b+c+d=0.
    U† has M(U†) = M(U)⁻¹, so logspec → (-d,-c,-b,-a) (already sorted).
    In monodromy: m0'=-d = m0+m1+m2, m1'=-c = -m2, m2'=-b = -m1.
    """
    return np.array([m[0] + m[1] + m[2], -m[2], -m[1]])


# Build dagger and rho monodromy for each grid point
dag_mono_points = np.array([true_dagger_mono(m) for m in mono_points])
rho_mono_points = np.array(
    [GateInvariants(tuple(m)).rho_reflect.monodromy for m in mono_points]
)

# Show that dagger ≠ rho in general
diff = np.max(np.abs(dag_mono_points - rho_mono_points), axis=1)
print(f"Points where dagger ≠ rho: {np.sum(diff > 1e-9)}/{len(diff)}")
print(f"Points where dagger = rho (c₃=0): {np.sum(diff < 1e-9)}/{len(diff)}")

m = mono_points
dag = dag_mono_points

# Reference: L(C, G, T)
ref_feas = qlr_feasible_bulk(m[i_c], m[i_g], m[i_t]).reshape(N, N, N)
print(
    f"\n{'C·G = T':20s}  L(C, G, T)          →  {ref_feas.sum():>8,} feasible  [REFERENCE]\n"
)

# All 12 rearrangements from C·G·T† = I, using TRUE dagger
rearrangements = [
    # Product = T
    ("G·C = T", "L(G, C, T)", m, i_g, m, i_c, m, i_t),
    # Product = G†
    ("C·T† = G†", "L(C, T†, G†)", m, i_c, dag, i_t, dag, i_g),
    ("T†·C = G†", "L(T†, C, G†)", dag, i_t, m, i_c, dag, i_g),
    # Product = C†
    ("G·T† = C†", "L(G, T†, C†)", m, i_g, dag, i_t, dag, i_c),
    ("T†·G = C†", "L(T†, G, C†)", dag, i_t, m, i_g, dag, i_c),
    # Product = G
    ("C†·T = G", "L(C†, T, G)", dag, i_c, m, i_t, m, i_g),
    ("T·C† = G", "L(T, C†, G)", m, i_t, dag, i_c, m, i_g),
    # Product = C
    ("G†·T = C", "L(G†, T, C)", dag, i_g, m, i_t, m, i_c),
    ("T·G† = C", "L(T, G†, C)", m, i_t, dag, i_g, m, i_c),
    # Product = T† (all inverted)
    ("G†·C† = T†", "L(G†, C†, T†)", dag, i_g, dag, i_c, dag, i_t),
    ("C†·G† = T†", "L(C†, G†, T†)", dag, i_c, dag, i_g, dag, i_t),
]

all_match = True
for label, qlr_form, pa, pi, ga, gi, ta, ti in rearrangements:
    f = qlr_feasible_bulk(pa[pi], ga[gi], ta[ti]).reshape(N, N, N)
    match = np.array_equal(ref_feas, f)
    if not match:
        all_match = False
    disagree = np.sum(ref_feas != f)
    only_ref = np.sum(ref_feas & ~f)
    only_perm = np.sum(~ref_feas & f)
    tag = (
        "✓ MATCH"
        if match
        else f"✗ {disagree:,} [{only_ref:,} lost, {only_perm:,} gained]"
    )
    print(f"{label:20s}  {qlr_form:20s}  →  {f.sum():>8,}  {tag}")

print(f"\n{'=' * 70}")
if all_match:
    print("CONFIRMED: Full S₃ permutation symmetry holds with TRUE DAGGER (†).")
else:
    print("S₃ broken. Investigating...")

Points where dagger ≠ rho: 152/181
Points where dagger = rho (c₃=0): 29/181

C·G = T               L(C, G, T)          →  1,629,001 feasible  [REFERENCE]

G·C = T               L(G, C, T)            →  1,629,001  ✓ MATCH
C·T† = G†             L(C, T†, G†)          →  1,629,001  ✓ MATCH
T†·C = G†             L(T†, C, G†)          →  1,629,001  ✓ MATCH
G·T† = C†             L(G, T†, C†)          →  1,629,001  ✓ MATCH
T†·G = C†             L(T†, G, C†)          →  1,629,001  ✓ MATCH
C†·T = G              L(C†, T, G)           →  1,629,001  ✓ MATCH
T·C† = G              L(T, C†, G)           →  1,629,001  ✓ MATCH
G†·T = C              L(G†, T, C)           →  1,629,001  ✓ MATCH
T·G† = C              L(T, G†, C)           →  1,629,001  ✓ MATCH
G†·C† = T†            L(G†, C†, T†)         →  1,629,001  ✓ MATCH
C†·G† = T†            L(C†, G†, T†)         →  1,629,001  ✓ MATCH

CONFIRMED: Full S₃ permutation symmetry holds with TRUE DAGGER (†).


## Results

**Full $S_3$ permutation symmetry holds** when using the **true dagger** $(\dagger)$ as the monodromy inverse.

### Key finding: dagger $\neq$ rho-reflect

The previous test used the rho-reflect as the monodromy inverse, which broke symmetry. The two operations differ by a $c_3$ sign flip in Weyl coordinates:

$$\dagger: (c_1, c_2, c_3) \to (1 - c_1, c_2, c_3) \qquad \rho: (c_1, c_2, c_3) \to (1 - c_1, c_2, -c_3)$$

They agree only on the $c_3 = 0$ face (CX–iSWAP plane). The rho-reflect is a Weyl group involution used internally by the LP solver to handle Weyl chamber sign ambiguity — it is **not** the physical gate inverse.

### The true dagger in monodromy

$$\dagger: (m_0, m_1, m_2) \to (m_0 + m_1 + m_2, \; -m_2, \; -m_1)$$

Derived from: logspec $(a,b,c,d) \to (-d,-c,-b,-a)$, which is the eigenvalue negation of $M(U^\dagger) = M(U)^{-1}$, already in canonical sorted order.

### Implication for segment rearrangement

Since $S_3$ symmetry holds with the true dagger, any feasible triple $L(C, G, T)$ guarantees feasibility of all 11 other rearrangements — provided we use $\dagger$ (not $\rho$) for inversions. This means segment rearrangement dispatch can freely permute roles without needing a separate QLR feasibility check.